# 00 – Datos: descarga y calidad
**Lab #2 – Derivados Financieros** · Punto 0 (rúbrica R4 – 3 %)

Objetivo: descargar los insumos y documentar fuente, fecha de descarga, limpieza, tratamiento de dividendos y coherencia temporal.

## 0.1 Descarga de precios (CSCO, ADI, PYPL, ^NDX)

In [1]:
import yfinance as yf                      # API (no oficial) de Yahoo Finance
import pandas as pd                        # manejo de tablas
import numpy as np                         # cálculos numéricos
from pathlib import Path                   # rutas que sirven en Windows y Linux
from datetime import date                  # para registrar la fecha de descarga

RAW = Path("../data/raw")                  # carpeta destino (relativa a notebooks/)
RAW.mkdir(parents=True, exist_ok=True)     # la crea si no existe

TICKERS = ["CSCO", "ADI", "PYPL", "^NDX"]  # 3 acciones + índice de mercado
INICIO = "2016-10-30"                      # 10 años: el máximo de la ventana permitida
FIN = "2026-10-31"                         # yfinance NO incluye 'end' → así entra el 30/10

# Reproducibilidad: los datos del laboratorio son la descarga del 01/11/2026 guardada en data/raw/.
# Con ACTUALIZAR = False el notebook lee esos CSV y NO vuelve a descargar (Yahoo puede ajustar datos).
ACTUALIZAR = False
if ACTUALIZAR or not (RAW / "adjclose_diario.csv").exists():
    df = yf.download(TICKERS, start=INICIO, end=FIN,
                     auto_adjust=False,    # trae Close y Adj Close por separado
                     actions=True,         # incluye dividendos y splits
                     progress=False)
    df["Close"].to_csv(RAW / "close_diario.csv")          # precio de cierre sin ajustar
    df["Adj Close"].to_csv(RAW / "adjclose_diario.csv")   # ajustado por dividendos y splits
    df["Dividends"].to_csv(RAW / "dividendos.csv")        # dividendos pagados por fecha
    FECHA_DESCARGA = date.today()
else:
    FECHA_DESCARGA = date(2026, 11, 1)                    # fecha de la descarga original

def leer_precios(nombre):                                 # lee un CSV de data/raw como tabla numérica
    t = pd.read_csv(RAW / nombre, index_col=0)
    t = t[pd.to_datetime(t.index, errors="coerce").notna()]   # quita filas de encabezado extra
    t.index = pd.to_datetime(t.index)
    return t.astype(float)

close = leer_precios("close_diario.csv")
adj = leer_precios("adjclose_diario.csv")
divs = leer_precios("dividendos.csv")
print("Datos de la descarga del", FECHA_DESCARGA)

Datos de la descarga del 2026-11-01


## 0.2 Tasa libre de riesgo (^TNX al 30/10/2026)

In [2]:
if ACTUALIZAR or not (RAW / "tnx.csv").exists():
    tnx = yf.download("^TNX", start="2026-10-01", end=FIN, auto_adjust=False, progress=False)
    tnx.to_csv(RAW / "tnx.csv")                            # serie cruda de ^TNX
tnx = pd.read_csv(RAW / "tnx.csv", header=[0, 1], index_col=0)    # columnas en dos niveles (campo, ticker)
tnx = tnx[pd.to_datetime(tnx.index, errors="coerce").notna()]     # quita la fila "Date" vacía
tnx.index = pd.to_datetime(tnx.index)
rf_pct = float(tnx["Close"].squeeze().astype(float).loc["2026-10-30"])   # ^TNX cotiza en puntos %
print(f"^TNX al 30/10/2026: {rf_pct:.3f} % → r_f = {rf_pct/100:.5f}")

^TNX al 30/10/2026: 4.929 % → r_f = 0.04929


## 0.3 Controles de calidad

In [3]:
print("Datos de la descarga del:", FECHA_DESCARGA)
print("Rango:", adj.index.min().date(), "→", adj.index.max().date())  # debe terminar el 2026-10-30
print("Observaciones:", len(adj))
print("=" * 30)
print("NaN por activo:\n", adj.isna().sum())

Datos de la descarga del: 2026-11-01
Rango: 2016-10-31 → 2026-10-30
Observaciones: 2514
NaN por activo:
 ADI     0
CSCO    0
PYPL    0
^NDX    0
dtype: int64


## 0.4 Limpieza y decisiones de datos
Controles adicionales sobre los precios descargados y decisiones de tratamiento (tomadas en el Punto 2):
ventana de 10 años, frecuencia diaria (principal) y mensual, precio Adj Close (retorno total) y retornos logarítmicos.

In [4]:
# 1) Integridad de las series
print("Fechas duplicadas:", close.index.duplicated().sum())
print("Fechas en orden cronológico:", close.index.is_monotonic_increasing)
print("Precios ≤ 0 (Close):", (close <= 0).sum().to_dict())
print("Mayor salto entre fechas consecutivas (días):", close.index.to_series().diff().dt.days.max(),
      "→ solo fines de semana y festivos de la bolsa de EE. UU.")

# 2) Movimientos diarios extremos (se conservan: son eventos reales de mercado, no errores)
r_abs = np.log(close).diff().abs()
extremos = pd.DataFrame({"Mayor movimiento diario (log)": r_abs.max(), "Fecha": r_abs.idxmax().dt.date})
display(extremos.style.format({"Mayor movimiento diario (log)": "{:.2%}"}))

# 3) Dividendos: eventos, último año y efecto en el retorno (Close vs. Adj Close)
k = 252
tabla_div = pd.DataFrame({
    "Pagos de dividendos (10 años)": (divs > 0).sum(),
    "Primer pago en la ventana": divs[divs > 0].apply(lambda s: s.first_valid_index()).dt.date,
    "Dividendos últimos 12 m (USD)": divs.loc["2025-10-31":].sum(),
    "Rend. dividendo (q)": divs.loc["2025-10-31":].sum() / close.iloc[-1],
    "Ret. anual precio (Close)": np.exp(np.log(close).diff().mean() * k) - 1,
    "Ret. anual total (Adj Close)": np.exp(np.log(adj).diff().mean() * k) - 1,
})
tabla_div["Aporte de dividendos (pp)"] = tabla_div["Ret. anual total (Adj Close)"] - tabla_div["Ret. anual precio (Close)"]
display(tabla_div.style.format({"Dividendos últimos 12 m (USD)": "{:,.2f}", "Rend. dividendo (q)": "{:.2%}",
        "Ret. anual precio (Close)": "{:.2%}", "Ret. anual total (Adj Close)": "{:.2%}",
        "Aporte de dividendos (pp)": "{:.2%}"}, na_rep="–"))
print("^NDX: Adj Close = Close en toda la serie →", np.allclose(adj["^NDX"], close["^NDX"]),
      "(índice de precio: no incorpora dividendos)")

Fechas duplicadas: 0
Fechas en orden cronológico: True
Precios ≤ 0 (Close): {'ADI': 0, 'CSCO': 0, 'PYPL': 0, '^NDX': 0}
Mayor salto entre fechas consecutivas (días): 4.0 → solo fines de semana y festivos de la bolsa de EE. UU.


,Mayor movimiento diario (log),Fecha
ADI,18.17%,2020-03-16
CSCO,14.77%,2022-05-19
PYPL,28.22%,2022-02-02
^NDX,13.00%,2020-03-16


,Pagos de dividendos (10 años),Primer pago en la ventana,Dividendos últimos 12 m (USD),Rend. dividendo (q),Ret. anual precio (Close),Ret. anual total (Adj Close),Aporte de dividendos (pp)
ADI,40,2016-11-30,4.29,1.03%,20.66%,22.91%,2.25%
CSCO,40,2017-01-04,1.67,1.60%,13.08%,16.44%,3.37%
PYPL,4,2025-11-19,0.56,0.98%,3.20%,3.31%,0.11%
^NDX,0,–,0.00,0.00%,19.51%,19.51%,0.00%


^NDX: Adj Close = Close en toda la serie → True (índice de precio: no incorpora dividendos)


## 0.5 Resumen de calidad de datos

| Aspecto | Tratamiento |
|---|---|
| **Fuente** | Yahoo Finance vía `yfinance` (precios diarios Close y Adj Close, dividendos y ^TNX), descarga del 01/11/2026. Los CSV de `data/raw/` son la base de todo el laboratorio (`ACTUALIZAR = False` evita re-descargar). |
| **Ventana** | 31/10/2016 – 30/10/2026: 10 años, el máximo permitido. PYPL cotiza desde jul-2015, así que las tres acciones tienen historia completa. |
| **Completitud** | 2,514 días hábiles por activo, 0 valores faltantes, 0 fechas duplicadas, 0 precios ≤ 0. Acciones e índice cotizan en el mismo calendario (NASDAQ), por lo que no hay fechas desalineadas. |
| **Valores extremos** | Se conservan: corresponden a eventos reales (COVID mar-2020; caídas de PYPL feb-2022 y de CSCO may-2022), no a errores de datos. |
| **Precio usado** | Adj Close (retorno total: incluye dividendos y splits) para retornos, matrices, Markowitz, VaR y betas. |
| **Dividendos** | CSCO y ADI pagaron 40 dividendos en la ventana; PYPL inició pagos en nov-2025. Rendimiento últimos 12 meses: CSCO 1.60 %, ADI 1.03 %, PYPL 0.98 %. Su aporte al retorno anual histórico (Adj Close − Close): CSCO ≈ 3.4 pp, ADI ≈ 2.3 pp, PYPL ≈ 0.1 pp. Para no contarlos dos veces, en el Punto 11 se descompone μ_total ≈ μ_precio + q. |
| **Índice de mercado** | ^NDX es un índice de precio (Adj Close = Close): no incluye dividendos. Esto se considera en el precio teórico del futuro (F₀ = S₀·e^{(r−q)T}). |
| **Tasa libre de riesgo** | ^TNX al 30/10/2026: 4.929 % anual (r_f = 0.04929). |
| **Coherencia temporal** | Precios, ^TNX y fundamentales tienen la misma fecha de corte (30/10/2026; fundamentales consultados el 01/11/2026). Las especificaciones del futuro se toman con la misma fecha de referencia (Punto 7). |
| **Frecuencias** | Diaria (k = 252) para P2, P3 y P5; mensual (k = 12, último precio de cada mes) para P4 y P8. Retornos logarítmicos. |